# 3.2 — Precision & Recall

Precision and recall answer two different questions about your model's predictions.

- **Precision** = TP / (TP + FP) — when you say YES, how often are you right?
- **Recall** = TP / (TP + FN) — of all actual YES cases, how many did you catch?

> **Support** = the count of actual samples per class in your test set. Small support = unreliable metrics for that class.

### The tradeoff:
- Improving precision usually hurts recall
- Improving recall usually hurts precision
- Which matters more depends entirely on your problem

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, classification_report

# Load and prepare Titanic — same pipeline as before
df = sns.load_dataset('titanic')
cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
df = df[cols].copy()
df['age'] = df['age'].fillna(df['age'].median())
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone'] = (df['family_size'] == 1).astype(int)
df['sex'] = LabelEncoder().fit_transform(df['sex'])
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

X = df.drop(columns=['survived'])
y = df['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

print("Setup complete.")

## Precision

Of all the passengers your model said survived — how many actually did?

`Precision = TP / (TP + FP)`

In [ ]:
precision = precision_score(y_test, y_pred)
print(f"Precision: {precision:.3f}")
print(f"\nInterpretation: When the model predicts 'survived'")
print(f"it is correct {precision:.1%} of the time.")

## Recall

Of all the passengers who actually survived — how many did the model correctly identify?

`Recall = TP / (TP + FN)`

In [ ]:
recall = recall_score(y_test, y_pred)
print(f"Recall: {recall:.3f}")
print(f"\nInterpretation: Of all passengers who actually survived")
print(f"the model caught {recall:.1%} of them.")
print(f"It missed {1-recall:.1%} of actual survivors.")

## Full Classification Report

Shows precision, recall, and support for both classes at once.

**Support** = count of actual samples per class in the test set.

In [ ]:
print(classification_report(y_test, y_pred,
      target_names=['Not survived', 'Survived']))

print("Support tells you how many actual samples per class:")
print(y_test.value_counts())

## The Precision-Recall Tradeoff

The model uses a threshold of 0.5 by default — if probability > 0.5, predict YES.
Changing this threshold shifts the balance between precision and recall.

In [ ]:
# Compare different thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

print(f"{'Threshold':>10}  {'Precision':>10}  {'Recall':>10}")
print("-" * 35)
for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    p = precision_score(y_test, y_t, zero_division=0)
    r = recall_score(y_test, y_t, zero_division=0)
    marker = " ← default" if t == 0.5 else ""
    print(f"{t:>10.1f}  {p:>10.3f}  {r:>10.3f}{marker}")

In [ ]:
# Plot the tradeoff curve
precisions, recalls, thresh = [], [], []
for t in np.arange(0.1, 0.95, 0.05):
    y_t = (y_prob >= t).astype(int)
    precisions.append(precision_score(y_test, y_t, zero_division=0))
    recalls.append(recall_score(y_test, y_t, zero_division=0))
    thresh.append(t)

plt.figure(figsize=(8, 4))
plt.plot(thresh, precisions, label='Precision', marker='.')
plt.plot(thresh, recalls, label='Recall', marker='.')
plt.axvline(x=0.5, color='gray', linestyle='--', label='Default threshold (0.5)')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision vs Recall at different thresholds')
plt.legend()
plt.tight_layout()
plt.show()

## When to Prioritise Which

| Problem | Prioritise | Why |
|---------|-----------|-----|
| Cancer detection | Recall | Missing a real case is dangerous |
| Spam filter | Precision | False alarms lose real emails |
| Fraud detection | Recall | Missing fraud is costly |
| Content recommendation | Precision | Bad recommendations annoy users |
| General classification | Both → use F1 | No clear cost difference |

> **Key insight:** Decide which metric matters before you train. Lowering the threshold raises recall. Raising it raises precision.

## Practice Task

In [ ]:
# Q1 — Print precision and recall for the Titanic model
# YOUR CODE HERE

# Q2 — Lower the threshold to 0.3 and recalculate precision and recall
# y_pred_low = (y_prob >= 0.3).astype(int)
# YOUR CODE HERE

# Q3 — Did recall go up or down when you lowered the threshold? Why?
# ANSWER:

# Q4 — For Titanic survival prediction, would you prioritise precision or recall?
# ANSWER:

# Q5 — What does the support column tell you about class balance in the test set?
# ANSWER: